# Gerando frases positivas a partir de todas as frases positivas reais.


In [ ]:
# Limpar memoria GPU
import torch
import gc

# 1. Remova as variáveis pesadas (substitua pelos nomes dos seus objetos)
del model
del tokenizer
del model

# 2. Limpe o lixo da memória RAM
gc.collect()

# 3. Esvazie o cache da GPU para o nvidia-smi mostrar como livre
torch.cuda.empty_cache()


# model_name = "Helsinki-NLP/opus-mt-en-ROMANCE"
# tokenizer = MarianTokenizer.from_pretrained(model_name)
# model = MarianMTModel.from_pretrained(model_name).to(device)


In [ ]:
# tranformar o arquivo csv complaints_with_sentiment.csv em um um dataframe para análises específicas de reclamações positivas.
df_complaints = pd.read_csv("complaints_with_sentiment.csv")
df_complaints.info()
df_complaints.head()

In [ ]:
# No data frame df_complaints separar os dados segundo a catagoria de sentiment_FinancialBERT iguais a "positive". para análises específicas de cada categoria.
df_positive = df_complaints[df_complaints['sentiment_FinancialBERT'] == 'positive']
df_positive.info()
df_positive.head()

In [ ]:
# usado no OLLAMA.
texts_positive = df_positive["narrative"].tolist()

In [ ]:
# transforar texts_positive em arquivo csv para análises específicas de reclamações positivas.
df_positive.to_csv("complaints_positive.csv", index=False)


In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

novas_frases = []

for _ in range(500):  # 500 vezes
    textos = generator(
        "The bank service was excellent and",
        max_length=50,
        num_return_sequences=10,   # gera 10 de uma vez
        temperature=0.9,           # criatividade
        top_k=50,
        top_p=0.95,
        do_sample=True
    )

    for t in textos:
        novas_frases.append(t["generated_text"])
frases_limpas = []

for frase in novas_frases:
    frase = frase.replace("The bank service was excellent and", "")
    frases_limpas.append(frase.strip())

import pandas as pd

df = pd.DataFrame(frases_limpas, columns=["texto"])
df.to_csv("frases_geradas.csv", index=False)
df = pd.read_csv("frases_geradas.csv")

In [ ]:
import pandas as pd
import ollama
from tqdm import tqdm
import time

def gerar_variacoes_local(frase_original, num_variacoes=3):
    """
    Gera variações de frases usando DeepSeek rodando localmente.
    """
    prompt = f"""
    Frase original: "{frase_original}"

    Gere {num_variacoes} versões diferentes desta frase, mantendo o mesmo significado,
    em Ingles. Responda apenas com as frases, uma por linha.
    """

    try:
        response = ollama.chat(
            model='deepseek-r1:8b',  # ou o modelo que você baixou
            messages=[
                {'role': 'system', 'content': 'Você é um assistente que gera variações de textos.'},
                {'role': 'user', 'content': prompt}
            ],
            options={
                'temperature': 0.8,
                'num_predict': 200
            }
        )

        resultado = response['message']['content'].strip()
        variacoes = [linha.strip() for linha in resultado.split('\n') if linha.strip()]

        # Limpar possíveis marcações como "1.", "2." etc.
        variacoes_limpas = []
        for v in variacoes:
            # Remove numerações iniciais
            import re
            v_limpa = re.sub(r'^\d+[\.\)]\s*', '', v)
            variacoes_limpas.append(v_limpa)

        return variacoes_limpas[:num_variacoes]

    except Exception as e:
        print(f"Erro: {e}")
        return [f"Erro: {frase_original}"] * num_variacoes

# ============================================
# Processamento similar ao anterior
# ============================================

df = pd.read_csv('complaints_positive.csv')

# Identifique sua coluna de texto
coluna_texto = df.columns[1]  # ou especifique pelo nome
print(f"Processando coluna: {coluna_texto}")

novas_frases = []
frases_originais = []

for idx, row in tqdm(df.head(50).iterrows(), total=50, desc="Testando"):  # Comece com 50 para testar
    frase = row[coluna_texto]
    variacoes = gerar_variacoes_local(frase, num_variacoes=9)

    for variacao in variacoes:
        novas_frases.append(variacao)
        frases_originais.append(frase)

df_resultado = pd.DataFrame({
    'frase_original': frases_originais,
    'frase_gerada': novas_frases
})

df_resultado.to_csv('frases_geradas_local.csv', index=False, encoding='utf-8')
print(f"✅ Geradas {len(df_resultado)} frases!")
#Erro: model requires more system memory (5.5 GiB) than is available (2.8 GiB) (status code: 500)